In [1]:
!pip install -q google-genai gradio pandas

In [2]:
import sqlite3
import pandas as pd
import re
import gradio as gr
from datetime import date, timedelta

DB_PATH = "/content/library.db"

def initialize_database():

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.executescript("""
    CREATE TABLE IF NOT EXISTS Books (
        book_id INTEGER PRIMARY KEY,
        title TEXT,
        author TEXT,
        category TEXT,
        total_copies INTEGER,
        available_copies INTEGER
    );

    CREATE TABLE IF NOT EXISTS Students (
        student_id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        email TEXT
    );

    CREATE TABLE IF NOT EXISTS Borrowing (
        borrow_id INTEGER PRIMARY KEY,
        student_id INTEGER,
        book_id INTEGER,
        borrow_date TEXT,
        due_date TEXT,
        return_date TEXT,
        FOREIGN KEY(student_id) REFERENCES Students(student_id),
        FOREIGN KEY(book_id) REFERENCES Books(book_id)
    );
    """)

    # Insert sample data only once
    if cursor.execute("SELECT COUNT(*) FROM Books").fetchone()[0] == 0:

        today = date.today()

        books = [
            (1, "Python Crash Course", "Eric Matthes",
             "Programming", 5, 3),

            (2, "Clean Code", "Robert Martin",
             "Programming", 4, 2),

            (3, "The Alchemist", "Paulo Coelho",
             "Fiction", 6, 5),

            (4, "Wings of Fire", "A. P. J. Abdul Kalam",
             "Biography", 3, 2),

            (5, "Introduction to Algorithms", "Thomas Cormen",
             "Computer Science", 2, 0),

            (6, "Atomic Habits", "James Clear",
             "Self-help", 4, 4)
        ]

        students = [
            (101, "Aarav Sharma", "Computer Science",
             "aarav@example.edu"),

            (102, "Isha Patil", "Information Technology",
             "isha@example.edu"),

            (103, "Rohan Deshmukh", "Computer Science",
             "rohan@example.edu"),

            (104, "Sara Khan", "Electronics",
             "sara@example.edu")
        ]

        borrowing = [
            (1, 101, 1,
             str(today - timedelta(days=12)),
             str(today - timedelta(days=5)), None),

            (2, 102, 2,
             str(today),
             str(today + timedelta(days=14)), None),

            (3, 103, 5,
             str(today - timedelta(days=20)),
             str(today - timedelta(days=10)), None),

            (4, 101, 3,
             str(today - timedelta(days=30)),
             str(today - timedelta(days=20)),
             str(today - timedelta(days=18)))
        ]

        cursor.executemany(
            "INSERT INTO Books VALUES (?,?,?,?,?,?)", books
        )

        cursor.executemany(
            "INSERT INTO Students VALUES (?,?,?,?)", students
        )

        cursor.executemany(
            "INSERT INTO Borrowing VALUES (?,?,?,?,?,?)", borrowing
        )

    conn.commit()
    conn.close()

initialize_database()

print("Library database created successfully!")

Library database created successfully!


In [3]:
from google.colab import userdata
from google import genai

try:
    API_KEY = userdata.get("GEMINI_API_KEY")
    client = genai.Client(api_key=API_KEY)
    print("Gemini connected successfully!")

except Exception:
    API_KEY = None
    client = None
    print("Demo mode enabled.")

Demo mode enabled.


In [5]:
SCHEMA = """
Books(book_id, title, author, category,
      total_copies, available_copies)

Students(student_id, name, department, email)

Borrowing(borrow_id, student_id, book_id,
          borrow_date, due_date, return_date)

Relationships:
Borrowing.student_id = Students.student_id
Borrowing.book_id = Books.book_id

Currently borrowed:
return_date IS NULL

Overdue:
return_date IS NULL AND due_date < date('now')
"""

def generate_sql(question):

    if client is None:
        return demo_sql(question)

    prompt = f"""
You are a Library Management Text-to-SQL Agent.

Convert the user's question into a valid SQLite SELECT query.

Database schema:
{SCHEMA}

Rules:
1. Generate only SELECT queries.
2. Never generate INSERT, UPDATE or DELETE.
3. Use correct SQLite syntax.
4. Use JOIN when required.
5. Return only SQL without markdown.

User question:
{question}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text.strip()

In [6]:
def demo_sql(question):

    q = question.lower()

    if "overdue" in q:

        return """
        SELECT Students.name, Books.title,
               Borrowing.due_date
        FROM Borrowing
        JOIN Students
        ON Borrowing.student_id = Students.student_id
        JOIN Books
        ON Borrowing.book_id = Books.book_id
        WHERE Borrowing.return_date IS NULL
        AND Borrowing.due_date < date('now');
        """

    elif "available" in q:

        return """
        SELECT title, author, category, available_copies
        FROM Books
        WHERE available_copies > 0;
        """

    elif "history" in q:

        match = re.search(r"\d+", question)

        if match:
            student_id = int(match.group())

            return f"""
            SELECT Students.name, Books.title,
                   Borrowing.borrow_date,
                   Borrowing.due_date,
                   Borrowing.return_date
            FROM Borrowing
            JOIN Students
            ON Borrowing.student_id = Students.student_id
            JOIN Books
            ON Borrowing.book_id = Books.book_id
            WHERE Students.student_id = {student_id};
            """

    elif "not returned" in q:

        return """
        SELECT Students.name, Books.title,
               Borrowing.due_date
        FROM Borrowing
        JOIN Students
        ON Borrowing.student_id = Students.student_id
        JOIN Books
        ON Borrowing.book_id = Books.book_id
        WHERE Borrowing.return_date IS NULL;
        """

    elif "category" in q:

        return """
        SELECT category,
               SUM(available_copies) AS available_books
        FROM Books
        GROUP BY category;
        """

    return "SELECT * FROM Books;"

In [7]:
def execute_sql(sql):

    sql = sql.strip().rstrip(";")

    # Only SELECT queries are allowed
    if not re.match(r"^SELECT\b", sql, re.IGNORECASE):
        raise ValueError("Only SELECT queries are allowed!")

    # Block dangerous SQL operations
    forbidden = [
        "INSERT", "UPDATE", "DELETE", "DROP",
        "ALTER", "CREATE", "PRAGMA", "ATTACH"
    ]

    for word in forbidden:

        if re.search(r"\b" + word + r"\b", sql, re.IGNORECASE):
            raise ValueError("Restricted SQL operation detected!")

    # Open database in read-only mode
    conn = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True
    )

    try:
        result = pd.read_sql_query(sql, conn)
    finally:
        conn.close()

    return result

In [ ]:
def ask_library(question):

    if not question.strip():
        return "Please enter a question.", pd.DataFrame()

    try:

        sql = generate_sql(question)

        result = execute_sql(sql)

        return sql, result

    except Exception as error:

        return str(error), pd.DataFrame()


with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown(
        "# 📚 Library Management Text-to-SQL Agent"
    )

    gr.Markdown(
        "Ask questions about books, students and borrowing records."
    )

    question = gr.Textbox(
        label="Enter your question",
        placeholder="Show all overdue books"
    )

    button = gr.Button(
        "Generate SQL and Run",
        variant="primary"
    )

    sql_output = gr.Code(
        label="Generated SQL",
        language="sql"
    )

    result_output = gr.Dataframe(
        label="Query Results"
    )

    button.click(
        fn=ask_library,
        inputs=question,
        outputs=[sql_output, result_output]
    )

app.launch(share=True, debug=True)

/tmp/ipykernel_533/1076673006.py:19: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9cb82181988ed05bcb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
